<a href="https://colab.research.google.com/github/TheSkyBiz/genai-multi-agent-experiments/blob/main/Multi_Agent_System_using_Gemini_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧑‍🤝‍🧑 Multi-Agent System using Gemini API

This project demonstrates how multiple AI agents can collaborate to complete a larger task.  
Each agent plays a specific role:
- **Planner** → Breaks the topic into subtopics
- **Researcher** → Finds information on each subtopic (using Gemini API)
- **Synthesizer** → Combines everything into a final structured report

Model used: **Gemini 2.0 Flash**

In [ ]:
!pip install -q google-generativeai
import google.generativeai as genai
from IPython.display import display, Markdown

In [ ]:
from IPython.display import display, Markdown

from google.colab import userdata
api_key = userdata.get('ga')

import os
import google.generativeai as genai

os.environ["GEMINI_API_KEY"] = api_key
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

MODEL_NAME = "gemini-2.0-flash"

# Utility function for nicer output
def pretty_print(text: str):
    display(Markdown(text))

## Agent 1: The Planner
Every good research project starts with a solid plan. A vague goal leads to an unclear outcome. The Planner’s only job is to prevent this by breaking down a broad topic into a list of specific, researchable questions.

In [ ]:
def planner(main_topic: str) -> list:
    prompt = f"""
    You are the Planner Agent.
    Break down the topic into 3-5 clear subtopics.
    Topic: {main_topic}
    """
    response = genai.GenerativeModel(MODEL_NAME).generate_content(prompt)
    subtopics = response.text.strip().split("\n")
    subtopics = [s.strip("-• ") for s in subtopics if s.strip()]
    return subtopics

## Agent 2: The Search Agent

Once the plan is set, we need an agent to execute it. The Search Agent takes one question at a time from the Planner’s list and uses a special tool, Google Search, to find the answer.

The role of this agent will be to gather raw, factual information for a single, specific question. It doesn’t write the report; it just finds the data.

In [ ]:
def researcher(subtopic: str) -> str:
    prompt = f"""
    You are the Researcher Agent.
    Collect useful, concise information about:
    {subtopic}
    Keep it short and factual.
    """
    response = genai.GenerativeModel(MODEL_NAME).generate_content(prompt)
    return response.text.strip()

## Agent 3: The Synthesizer

With a pile of research notes, we need someone to make sense of it all. The Synthesizer’s job is to take all the fragmented answers from the Search Agent and weave them into a single, well-structured, and coherent report.

The role of this agent is to synthesize information, not to find it. It is explicitly stated that only the data provided by the Search Agent should be used.

In [ ]:
def synthesizer(topic: str, researched_data: dict) -> str:
    combined_text = "\n".join(
        [f"{subtopic}: {content}" for subtopic, content in researched_data.items()]
    )

    prompt = f"""
    You are the Synthesizer Agent.
    Write a structured final report on: {topic}.
    Use the following researched notes:
    {combined_text}
    The report should be clear and well-organized.
    """
    response = genai.GenerativeModel(MODEL_NAME).generate_content(prompt)
    return response.text.strip()

## The Conductor: The main() Orchestrator

Now, we will build the main function that directs the entire process, ensuring each agent performs at the right time and that their work flows seamlessly from one to the next:

In [ ]:
def run_multi_agent():
    topic = input("Enter the main topic: ")

    print("\n[Planner Agent Working...]")
    subtopics = planner(topic)
    print("Subtopics:", subtopics)

    researched_data = {}
    for sub in subtopics:
        print(f"\n[Researcher Agent Working on: {sub}]")
        researched_data[sub] = researcher(sub)

    print("\n[Synthesizer Agent Working...]")
    final_report = synthesizer(topic, researched_data)

    print("\n======= Final Report =======")
    pretty_print(final_report)

run_multi_agent()

Enter the main topic: Reinforcement Learning in Neural Networks

[Planner Agent Working...]
Subtopics: ['Okay, here\'s a breakdown of "Reinforcement Learning in Neural Networks" into 5 clear subtopics:', '1.  **Fundamentals of Reinforcement Learning:** This covers the core principles of RL, including:', '*   **Agents, Environments, States, Actions, and Rewards:** Defining these key components.', '*   **Markov Decision Processes (MDPs):** The mathematical framework underlying RL.', '*   **Value Functions and Policies:** Understanding how agents learn optimal strategies.', "*   **Exploration vs. Exploitation:** Balancing learning new information with using what's already known.", '2.  **Neural Networks as Function Approximators in RL:** This focuses on how neural networks are used to represent value functions or policies in RL:', '*   **Replacing Lookup Tables:** How neural networks overcome the limitations of traditional tabular methods.', '*   **Types of Neural Network Architectures:**

## Final Report: Reinforcement Learning in Neural Networks

**1. Introduction:**

Reinforcement Learning (RL) is a paradigm for training agents to make sequential decisions in an environment to maximize cumulative reward.  The agent learns through trial and error, receiving feedback in the form of rewards (or penalties) for its actions.  While traditional RL methods often rely on tabular representations, these become infeasible for large or continuous state and action spaces. Neural Networks (NNs) provide a powerful solution as function approximators, enabling RL to tackle complex, real-world problems. This report provides a structured overview of Reinforcement Learning in Neural Networks, exploring its core concepts, algorithms, training techniques, applications, and challenges.

**2. Fundamentals of Reinforcement Learning:**

Reinforcement Learning is fundamentally about learning optimal behavior through interaction.  The agent learns a *policy* that maps states to actions, aiming to maximize cumulative reward over time.

*   **2.1 Key Components:**
    *   **Agent:** The decision-maker that perceives the environment and takes actions.
    *   **Environment:** The world the agent interacts with, providing states and rewards.
    *   **State:** A description of the environment's current situation, as perceived by the agent.
    *   **Action:** A choice made by the agent that affects the environment.
    *   **Reward:** A scalar value indicating the desirability of a particular action taken in a particular state.
    *   **Policy:** A strategy that dictates the agent's action in each state.  Can be deterministic or stochastic.
    *   **Value Function:** Estimates the expected cumulative reward from a given state (or state-action pair) following a particular policy.
*   **2.2 Markov Decision Processes (MDPs):** The mathematical framework underlying RL, defining the environment as a tuple `(S, A, P, R, γ)`: states, actions, transition probabilities, reward function, and discount factor.
*   **2.3 Value Functions and Policies:** Value functions (State-Value V(s) and Action-Value Q(s,a)) predict future rewards and guide the learning process.  Policies map states to actions, and the goal is to find the *optimal* policy π* that maximizes the expected cumulative reward.
*   **2.4 Exploration vs. Exploitation:** A key challenge is balancing exploring new options to gather information and exploiting known actions that currently yield high rewards. Strategies like epsilon-greedy, UCB, and Thompson Sampling are used to manage this trade-off.

**3. Neural Networks as Function Approximators in RL:**

Neural Networks serve as powerful function approximators in RL, particularly for handling large and continuous state and action spaces where traditional tabular methods fall short.

*   **3.1 Replacing Lookup Tables:** NNs overcome the limitations of lookup tables by learning continuous functions that approximate input-output mappings, providing generalization to unseen states, reduced memory footprint, adaptability, and automated feature extraction.
*   **3.2 Types of Neural Network Architectures:**
    *   **Multi-Layer Perceptron (MLP):** Suitable for simple state representations and value function approximation.
    *   **Convolutional Neural Networks (CNNs):** Excellent for environments with visual input, learning features directly from raw pixel data.
    *   **Recurrent Neural Networks (RNNs):** Effective for Partially Observable Markov Decision Processes (POMDPs) and environments with temporal dependencies.
    *   **Transformers:** Emerging architectures for RL, capable of handling long-range dependencies and complex sequential patterns.
*   **3.3 Function Approximation Challenges:**  Challenges include instability, convergence problems (due to poor learning rates or non-convex loss landscapes), overfitting, and the curse of dimensionality. Addressing these issues requires careful design and regularization.

**4. Deep Reinforcement Learning Algorithms:**

Deep Reinforcement Learning (DRL) algorithms combine the representation learning capabilities of Deep Learning with the decision-making framework of Reinforcement Learning.

*   **4.1 Q-Learning and Deep Q-Networks (DQNs):** DQNs use deep neural networks to approximate the Q-function, enabling RL to handle high-dimensional state spaces. Key features include experience replay and target networks to stabilize training.
*   **4.2 Policy Gradient Methods:** Directly optimize the policy using gradient ascent.
    *   **REINFORCE:** A Monte Carlo method with high variance.
    *   **Actor-Critic Methods (A2C, A3C):** Combine a policy network (actor) with a value function network (critic) to reduce variance and improve learning.
    *   **Proximal Policy Optimization (PPO):** Improves stability and sample efficiency by limiting policy changes at each update.
*   **4.3 Actor-Critic Methods:** Combine value-based and policy-based approaches, utilizing an "actor" (policy network) and a "critic" (value network) to provide feedback and improve policy learning.
*   **4.4 Algorithm Selection:** Considerations for choosing the right algorithm include the problem type, data characteristics, performance requirements, computational resources, and the trade-off between interpretability and performance.

**5. Training and Optimization Techniques in Deep RL:**

Training Deep RL models requires specific techniques to ensure stability and efficiency.

*   **5.1 Experience Replay:** Stores past experiences in a replay buffer and samples them randomly during training, breaking correlations and improving data efficiency.
*   **5.2 Target Networks:** Uses separate networks to stabilize Q-value updates by reducing the variance in target values.
*   **5.3 Reward Shaping:** Designing reward functions to guide the agent's learning process, although careful design is required to avoid unintended consequences.
*   **5.4 Hyperparameter Tuning:** Optimizing learning rates, batch sizes, and other parameters is crucial for achieving optimal performance. Techniques like grid search, random search, and Bayesian optimization are employed.

**6. Applications and Challenges:**

Reinforcement Learning with Neural Networks has demonstrated impressive results in various domains but faces significant challenges.

*   **6.1 Applications:**
    *   **Gaming:** Superhuman performance in Atari games, Go, and other strategic games.
    *   **Robotics:** Robot control, manipulation, and navigation in complex environments.
    *   **Autonomous Driving:** Lane keeping, navigation, and traffic management.
    *   **Finance:** Algorithmic trading, portfolio optimization, and risk management.
    *   **Healthcare:** Personalized treatment plans and drug discovery.
*   **6.2 Challenges:**
    *   **Sample Efficiency:** Requires vast amounts of data for training.
    *   **Generalization:** Models often struggle to generalize to new environments.
    *   **Safety Concerns:** Ensuring safe exploration and avoiding dangerous actions.
    *   **Interpretability:** Lack of transparency in decision-making processes.
*   **6.3 Current Research Directions:**
    *   **Meta-Learning:** Developing agents that can quickly adapt to new environments.
    *   **Transfer Learning:** Transferring knowledge between different environments.
    *   **Hierarchical Reinforcement Learning:** Decomposing complex tasks into hierarchies of sub-tasks.

**7. Conclusion:**

Reinforcement Learning in Neural Networks is a rapidly evolving field with the potential to revolutionize numerous applications. While significant challenges remain, ongoing research into sample efficiency, generalization, safety, and interpretability promises to unlock even greater capabilities, enabling the development of intelligent agents capable of solving complex real-world problems.